# Luminos Health TCOC — Build Execution Evidence

Collects text evidence that the TCOC build ran end to end (pipeline, gold tables, ML model, serving endpoint, Lakebase, app) and writes it as Markdown files into this `evidence/` folder so it can be committed and read as plain text. Only aggregated results are written — no member- or patient-level rows. Workspace host and user emails are replaced with placeholders.

In [0]:
%pip install -q lightgbm psycopg2-binary tabulate "databricks-sdk>=0.118.0"
dbutils.library.restartPython()

In [0]:
import json
import re
from datetime import datetime, timezone
from pathlib import Path

import mlflow
import pandas as pd
from databricks.sdk import WorkspaceClient
from mlflow import MlflowClient

w = WorkspaceClient()
mlflow.set_registry_uri("databricks-uc")
mlflow_client = MlflowClient(registry_uri="databricks-uc")

CATALOG = "phanitha_test_catalog"
PIPELINE_ID = "24877043-6556-40fa-8603-442ca50b2a20"
MODEL_NAME = f"{CATALOG}.tcoc_ml.cost_forecast_model"
ENDPOINT_NAME = "cost_forecast_model"
APP_NAME = "luminos-health-tcoc"
LAKEBASE_ENDPOINT = "projects/tcoc-app/branches/production/endpoints/primary"

notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
EVIDENCE_DIR = Path("/Workspace" + str(Path(notebook_path).parent))
REPO_DIR = EVIDENCE_DIR.parent
RUN_TS = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M UTC")

HOST = (w.config.host or "").replace("https://", "").rstrip("/")


def scrub(text: str) -> str:
    text = str(text)
    if HOST:
        text = text.replace(HOST, "<workspace-host>")
    text = re.sub(r"[\w.+-]+@[\w-]+\.[\w.]+", "<user-email>", text)
    text = re.sub(r"ep-[a-z0-9-]+\.database\.[a-z0-9-]+\.cloud\.databricks\.com", "<lakebase-host>", text)
    return text


def md_table(df: pd.DataFrame) -> str:
    if df is None or df.empty:
        return "_(no rows)_"
    return scrub(df.to_markdown(index=False))


def sql_df(query: str) -> pd.DataFrame:
    return spark.sql(query).toPandas()


def write_evidence(filename: str, title: str, sections: list) -> Path:
    lines = [f"# {title}", "", f"_Generated by `evidence/collect_build_evidence` on {RUN_TS}._", ""]
    for heading, body in sections:
        lines += [f"## {heading}", "", body, ""]
    path = EVIDENCE_DIR / filename
    path.write_text(scrub("\n".join(lines)))
    print(f"Wrote {path.relative_to(REPO_DIR)} ({path.stat().st_size:,} bytes)")
    return path


print(f"Evidence folder: {EVIDENCE_DIR}")
print(f"Run timestamp: {RUN_TS}")

In [0]:
pipeline = w.pipelines.get(PIPELINE_ID)
updates = w.pipelines.list_updates(PIPELINE_ID, max_results=10).updates or []
updates_df = pd.DataFrame([
    {
        "update_id": u.update_id,
        "state": u.state.value if u.state else None,
        "created_utc": datetime.fromtimestamp(u.creation_time / 1000, timezone.utc).strftime("%Y-%m-%d %H:%M:%S") if u.creation_time else None,
    }
    for u in updates
])

tables_df = sql_df(f"""
    SELECT table_schema, table_name, table_type
    FROM {CATALOG}.information_schema.tables
    WHERE table_schema IN ('tcoc_bronze', 'tcoc_silver', 'tcoc_gold')
      AND table_type IN ('STREAMING_TABLE', 'MATERIALIZED_VIEW')
    ORDER BY table_schema, table_name
""")
counts = []
for row in tables_df.itertuples(index=False):
    n = spark.sql(f"SELECT COUNT(*) AS n FROM {CATALOG}.{row.table_schema}.{row.table_name}").first()["n"]
    counts.append({"schema": row.table_schema, "table": row.table_name, "type": row.table_type, "row_count": n})
counts_df = pd.DataFrame(counts)

summary = (
    f"* Pipeline: **{pipeline.name}** (`{PIPELINE_ID}`)\n"
    f"* State: `{pipeline.state.value if pipeline.state else 'n/a'}`\n"
    f"* Datasets materialized: {len(counts_df)} "
    f"({(counts_df['schema'] == 'tcoc_bronze').sum()} bronze, {(counts_df['schema'] == 'tcoc_silver').sum()} silver, {(counts_df['schema'] == 'tcoc_gold').sum()} gold)\n"
    f"* Total rows across datasets: {int(counts_df['row_count'].sum()):,}"
)
write_evidence("01_pipeline_run.md", "Lakeflow Spark Declarative Pipeline — Run Evidence", [
    ("Summary", summary),
    ("Recent pipeline updates", md_table(updates_df)),
    ("Row counts per dataset (live query)", md_table(counts_df)),
])
display(updates_df)
display(counts_df)

In [0]:
gold = f"{CATALOG}.tcoc_gold"
queries = {
    "Executive KPIs (`tcoc_executive_kpis`)": f"SELECT * FROM {gold}.tcoc_executive_kpis",
    "Monthly PMPM by line of business (dashboard dataset `ds_pmpm_lob`)": f"""
        SELECT CAST(report_month AS DATE) AS report_month, line_of_business,
               ROUND(SUM(total_paid) / SUM(active_members), 2) AS pmpm
        FROM {gold}.tcoc_pmpm_trends GROUP BY 1, 2 ORDER BY 1, 2""",
    "Cost by risk tier and line of business (dashboard dataset `ds_risk_tier`)": f"""
        SELECT cost_risk_tier, line_of_business, COUNT(*) AS member_count,
               ROUND(SUM(total_paid), 0) AS total_cost
        FROM {gold}.tcoc_member_summary GROUP BY 1, 2 ORDER BY 1, 2""",
    "Total paid by service category (dashboard dataset `ds_svc_paid`)": f"""
        SELECT procedure_category, ROUND(SUM(total_paid), 0) AS total_paid
        FROM {gold}.tcoc_service_category_costs GROUP BY 1 ORDER BY 2 DESC""",
    "In-network vs out-of-network paid by LOB (dashboard dataset `ds_net_lob`)": f"""
        SELECT line_of_business, network_status, ROUND(SUM(total_paid), 0) AS total_paid
        FROM {gold}.tcoc_provider_network_analysis GROUP BY 1, 2 ORDER BY 1, 2""",
    "Top 10 states by PMPM (`tcoc_geographic_analysis`)": f"""
        SELECT member_state, ROUND(SUM(total_paid) / SUM(active_members), 2) AS pmpm,
               SUM(claim_count) AS claims
        FROM {gold}.tcoc_geographic_analysis GROUP BY 1 ORDER BY 2 DESC LIMIT 10""",
}
sections = []
for title, q in queries.items():
    df = sql_df(q)
    sections.append((title, f"```sql\n{q.strip()}\n```\n\nRows returned: {len(df)}\n\n{md_table(df)}"))
    print(f"{title}: {len(df)} rows")
write_evidence("02_gold_and_dashboard_queries.md", "Gold Layer and Dashboard Query Results", sections)

In [0]:
versions = sorted(mlflow_client.search_model_versions(f"name='{MODEL_NAME}'"), key=lambda v: int(v.version))
version_rows = []
for v in versions:
    run = mlflow_client.get_run(v.run_id)
    version_rows.append({
        "version": v.version,
        "status": v.status,
        "created_utc": datetime.fromtimestamp(v.creation_timestamp / 1000, timezone.utc).strftime("%Y-%m-%d %H:%M"),
        "run_name": run.info.run_name,
        "mape": round(run.data.metrics.get("mape", float("nan")), 4),
        "mae": round(run.data.metrics.get("mae", float("nan")), 2),
        "rmse": round(run.data.metrics.get("rmse", float("nan")), 2),
        "train_rows": run.data.params.get("train_rows"),
        "valid_rows": run.data.params.get("valid_rows"),
        "feature_count": run.data.params.get("feature_count"),
    })
versions_df = pd.DataFrame(version_rows)

latest = versions[-1]
latest_run = mlflow_client.get_run(latest.run_id)
exp_runs = mlflow.search_runs(
    experiment_ids=[latest_run.info.experiment_id],
    filter_string="tags.project = 'luminos-health-tcoc'",
    order_by=["start_time DESC"],
    max_results=10,
)
comparison_df = exp_runs[["tags.mlflow.runName", "start_time", "metrics.mape", "metrics.mae", "metrics.rmse"]].rename(columns={
    "tags.mlflow.runName": "run_name", "metrics.mape": "mape", "metrics.mae": "mae", "metrics.rmse": "rmse"})
comparison_df["start_time"] = comparison_df["start_time"].dt.strftime("%Y-%m-%d %H:%M")

imp_path = mlflow.artifacts.download_artifacts(run_id=latest.run_id, artifact_path="artifacts/lightgbm_feature_importance.csv")
importance_df = pd.read_csv(imp_path).head(15)

val_path = mlflow.artifacts.download_artifacts(run_id=latest.run_id, artifact_path="validation/validation_predictions_lightgbm.csv")
val_df = pd.read_csv(val_path)
val_df["abs_pct_error"] = (val_df["predicted_total_paid"] - val_df["actual_total_paid"]).abs() / val_df["actual_total_paid"]
val_summary = (
    val_df.groupby(["service_month", "line_of_business"], as_index=False)
    .agg(actual_total_paid=("actual_total_paid", "sum"), predicted_total_paid=("predicted_total_paid", "sum"), segments=("plan_type", "count"))
)
val_summary["pct_error"] = ((val_summary["predicted_total_paid"] / val_summary["actual_total_paid"]) - 1).round(4)
val_summary[["actual_total_paid", "predicted_total_paid"]] = val_summary[["actual_total_paid", "predicted_total_paid"]].round(0).astype("int64")

forecast_df = sql_df(f"""
    SELECT CAST(forecast_month AS DATE) AS forecast_month, line_of_business, model_version,
           COUNT(*) AS segments, ROUND(SUM(predicted_paid), 0) AS predicted_paid,
           ROUND(SUM(lower_bound), 0) AS lower_bound, ROUND(SUM(upper_bound), 0) AS upper_bound
    FROM {CATALOG}.tcoc_gold.tcoc_cost_forecast GROUP BY 1, 2, 3 ORDER BY 1, 2""")

write_evidence("03_ml_model_and_forecast.md", "Cost Forecast Model — Training, Registry and Forecast Output", [
    ("Registered model versions (Unity Catalog)", f"Model: `{MODEL_NAME}`\n\n{md_table(versions_df)}"),
    ("Experiment runs: LightGBM vs linear baseline", md_table(comparison_df)),
    (f"Top 15 LightGBM feature importances (version {latest.version})", md_table(importance_df)),
    (f"Holdout validation Oct–Dec 2025: actual vs predicted by month and LOB (version {latest.version})",
     f"Segment-level rows: {len(val_df)}; mean absolute % error: {val_df['abs_pct_error'].mean():.4f}\n\n{md_table(val_summary)}"),
    ("Forecast output (`tcoc_gold.tcoc_cost_forecast`)", f"Total rows: {int(forecast_df['segments'].sum()):,}\n\n{md_table(forecast_df)}"),
])
display(versions_df)
display(forecast_df)

In [0]:
ep = w.serving_endpoints.get(ENDPOINT_NAME)
served = [f"{e.entity_name} v{e.entity_version}" for e in (ep.config.served_entities or [])] if ep.config else []
state_txt = f"ready=`{ep.state.ready.value if ep.state and ep.state.ready else 'n/a'}`, config_update=`{ep.state.config_update.value if ep.state and ep.state.config_update else 'n/a'}`"

model_uri = f"models:/{MODEL_NAME}/{served[0].split(' v')[-1]}" if served else f"models:/{MODEL_NAME}/{latest.version}"
local_model_dir = mlflow.artifacts.download_artifacts(artifact_uri=model_uri)
input_example = mlflow.models.Model.load(local_model_dir).load_input_example(local_model_dir)
records = json.loads(input_example.to_json(orient="records"))

try:
    response = w.serving_endpoints.query(name=ENDPOINT_NAME, dataframe_records=records)
    predictions = response.predictions
    score_df = pd.DataFrame(records)[["line_of_business", "plan_type", "member_state", "year_num", "month_num", "paid_lag_1", "paid_rolling_3m_avg"]]
    score_df["predicted_total_paid"] = [round(float(p), 2) for p in predictions]
    result = f"Request: {len(records)} records via `dataframe_records`.\n\n{md_table(score_df)}"
except Exception as e:
    result = f"Scoring request failed: `{type(e).__name__}: {str(e)[:500]}`"

write_evidence("04_serving_endpoint_scoring.md", "Model Serving Endpoint — Live Scoring Evidence", [
    ("Endpoint", f"* Name: `{ENDPOINT_NAME}`\n* Served entities: {', '.join(served) or 'n/a'}\n* State: {state_txt}"),
    ("Live scoring response", result),
])
print(result[:2000])

In [0]:
from psycopg2 import connect

lb_ep = w.postgres.get_endpoint(name=LAKEBASE_ENDPOINT)
token = w.postgres.generate_database_credential(endpoint=lb_ep.name).token
conn = connect(host=lb_ep.status.hosts.host, port=5432, dbname="databricks_postgres",
               user=w.current_user.me().user_name, password=token, sslmode="require")
cur = conn.cursor()
cur.execute("""SELECT table_name FROM information_schema.tables
               WHERE table_schema = 'tcoc_gold' ORDER BY table_name""")
lb_rows = []
for (tbl,) in cur.fetchall():
    cur.execute(f'SELECT COUNT(*) FROM tcoc_gold."{tbl}"')
    lb_rows.append({"lakebase_table": f"tcoc_gold.{tbl}", "row_count": cur.fetchone()[0]})
conn.close()
lakebase_df = pd.DataFrame(lb_rows)

app = w.apps.get(APP_NAME)
app_info = (
    f"* App: `{APP_NAME}`\n"
    f"* App status: `{app.app_status.state.value if app.app_status and app.app_status.state else 'n/a'}` — {app.app_status.message if app.app_status else ''}\n"
    f"* Compute status: `{app.compute_status.state.value if app.compute_status and app.compute_status.state else 'n/a'}`\n"
    f"* Active deployment: `{app.active_deployment.deployment_id if app.active_deployment else 'n/a'}` — "
    f"`{app.active_deployment.status.state.value if app.active_deployment and app.active_deployment.status and app.active_deployment.status.state else 'n/a'}`"
)
write_evidence("05_lakebase_and_app.md", "Lakebase Serving Layer and Databricks App — Status Evidence", [
    ("Lakebase production branch: `tcoc_gold` tables read by the app", md_table(lakebase_df)),
    ("Databricks App", app_info),
])
display(lakebase_df)
print(scrub(app_info))

In [0]:
import base64
import io
from databricks.sdk.service.workspace import ExportFormat

ml_nb_path = str(REPO_DIR / "ml" / "tcoc_cost_forecast_model").replace("/Workspace", "", 1)
raw = w.workspace.export(ml_nb_path, format=ExportFormat.JUPYTER).content
nb = json.loads(base64.b64decode(raw).decode())

log = []
code_idx = 0
for cell in nb["cells"]:
    if cell["cell_type"] != "code":
        continue
    code_idx += 1
    title = cell.get("metadata", {}).get("application/vnd.databricks.v1+cell", {}).get("title") or f"Cell {code_idx}"
    log.append(f"### {code_idx}. {title}")
    outputs = cell.get("outputs", [])
    if not outputs:
        log.append("_(no output captured)_")
    for out in outputs:
        otype = out.get("output_type")
        if otype == "stream":
            text = "".join(out.get("text", []))
        elif otype in ("execute_result", "display_data"):
            data = out.get("data", {})
            if "text/plain" in data:
                text = "".join(data["text/plain"])
            elif "text/html" in data:
                html = "".join(data["text/html"])
                rows = [
                    [re.sub(r"<[^>]+>", "", c).strip() for c in re.findall(r"<t[hd][^>]*>(.*?)</t[hd]>", r, re.S)]
                    for r in re.findall(r"<tr[^>]*>(.*?)</tr>", html, re.S)
                ]
                rows = [r for r in rows if r]
                if rows:
                    header, body = rows[0], rows[1:]
                    text = f"[table: {len(body)} rows; first {min(15, len(body))} shown]\n" + pd.DataFrame(body[:15], columns=header).to_markdown(index=False)
                else:
                    text = re.sub(r"\s+", " ", re.sub(r"<[^>]+>", " ", html)).strip()
            elif any(k.startswith("image/") for k in data):
                text = "[chart rendered — see tables in the evidence files for the underlying values]"
            else:
                text = f"[{', '.join(data.keys())} output]"
        elif otype == "error":
            text = f"{out.get('ename')}: {out.get('evalue')}"
        else:
            text = f"[{otype}]"
        text = re.sub(r"\x1b\[[0-9;]*m", "", text).strip()
        if len(text) > 4000:
            text = text[:4000] + "\n... [truncated]"
        log.append("```text\n" + text + "\n```")
    log.append("")

captured = sum(1 for c in nb["cells"] if c["cell_type"] == "code" and c.get("outputs"))
write_evidence("06_ml_notebook_run_log.md", "Cost Forecast Notebook — Cell Output Log", [
    ("Source", f"Notebook `ml/tcoc_cost_forecast_model.ipynb`; {captured} of {code_idx} code cells had captured output at export time."),
    ("Cell outputs", "\n".join(log)),
])

In [0]:
files = sorted(p.name for p in EVIDENCE_DIR.glob("0*.md"))
index = "\n".join([
    "# Build Execution Evidence — Luminos Health TCOC",
    "",
    f"Generated on {RUN_TS} by running `evidence/collect_build_evidence` against the live workspace.",
    "All values come from live queries, MLflow, the Model Serving endpoint, Lakebase and the Apps API. All source data is synthetic, and only aggregated results are included (no member- or patient-level rows).",
    "",
    "| File | Evidence |",
    "| --- | --- |",
    "| `01_pipeline_run.md` | Lakeflow Spark Declarative Pipeline update history and row counts for all bronze/silver/gold datasets |",
    "| `02_gold_and_dashboard_queries.md` | Results of the gold-layer queries that back the AI/BI dashboard |",
    "| `03_ml_model_and_forecast.md` | Registered model versions, LightGBM vs baseline metrics, feature importances, holdout validation, forecast output |",
    "| `04_serving_endpoint_scoring.md` | Live scoring request and response from the Model Serving endpoint |",
    "| `05_lakebase_and_app.md` | Lakebase table row counts read by the app, and Databricks App deployment status |",
    "| `06_ml_notebook_run_log.md` | Text log of the forecast notebook's cell outputs |",
    "",
    "Files present: " + ", ".join(f"`{f}`" for f in files),
])
(EVIDENCE_DIR / "README.md").write_text(scrub(index) + "\n")
print(index)